# Training Notebook
This notebook runs local queue-environment smoke tests, PPO training, and the combined benchmark using the project modules.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "baselines").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

In [ ]:
from config.base_config import hyperparams_config

cfg = hyperparams_config()
cfg

## Optional: Heuristic Benchmark Smoke Test\n
Run this first to verify env behavior quickly before PPO.

In [ ]:
from envs.queue_env import QueueEnv
from baselines.heuristics import build_default_baselines, evaluate_baselines

env = QueueEnv(max_jobs=cfg["max_jobs"], max_steps=cfg["max_steps"])
results = evaluate_baselines(
    build_default_baselines(),
    env,
    n_episodes=cfg["episodes"],
    base_seed=cfg["seed"],
)

for r in results:
    print(r["name"], "reward=", round(r["mean_reward"], 2), "drops=", round(r["mean_dropped_jobs"], 2))

## Train PPO
By default this uses `total_timesteps` from `config/config.ini`.

In [ ]:
from baselines.train_ppo import PPOTrainer

trainer = PPOTrainer()

# Optional quick test run: uncomment to shorten training
# trainer.config["total_timesteps"] = 50_000

trainer.train()
saved_path = trainer.save("ppo_queue_notebook")
print("Saved PPO model to", saved_path)

## Optional: Train Custom PPO
Run this only if you want to compare your own PPO implementation against SB3 and the heuristics.


In [ ]:
from baselines.ppo_agent import PPOAgent

custom_trainer = PPOAgent()

# Optional quick test run: uncomment to shorten training
# custom_trainer.config["total_timesteps"] = 50_000

custom_trainer.train()
custom_saved_path = custom_trainer.save("custom_ppo_notebook")
print("Saved custom PPO model to", custom_saved_path)


## Evaluate and Benchmark
Load any saved PPO models and compare them against the heuristic baselines.


In [ ]:
from baselines.eval_all import main

custom_path = custom_saved_path if "custom_saved_path" in globals() else None

main(
    train_ppo=False,
    model_path=saved_path,
    train_custom_ppo=False,
    custom_model_path=custom_path,
)
